In [3]:
!uv pip install unsloth --torch-backend=auto

Using Python 3.12.12 environment at: /usr
Audited 1 package in 130ms


In [45]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
Memory: 15.6 GB


In [47]:
import os
import torch, time, random, json, re
import numpy as np
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import (
    TrainingArguments,
    default_data_collator,
    EarlyStoppingCallback,
)
from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [48]:
class Config:
    SEED = 42
    MODEL_NAME = "mistralai/Mistral-7B-v0.1"

    # Dataset Sizes
    TRAIN_SIZE = 2000
    TEST_SIZE = 200
    GEN_SIZE = 100
    FORGET_SIZE = 100

    # Training Params
    MAX_SEQ_LEN = 512
    CONTEXT_LEN = 800
    EPOCHS = 1
    BATCH_SIZE = 4
    LR = 2e-4

    # LoRA Params
    LORA_R = 8
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.0 # Unsloth optimized dropout

    # Paths
    OUTPUT_DIR = "./lora_model"
    RESULTS_FILE = "results.json"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

In [49]:
def load_squad_splits():
    print("Loading SQuAD Dataset splits...")
    return {
        'train': load_dataset("squad", split=f"train[:{Config.TRAIN_SIZE}]"),
        'test': load_dataset("squad", split=f"validation[:{Config.TEST_SIZE}]"),
        'gen': load_dataset("squad", split=f"validation[{Config.TEST_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE}]"),
        'forget': load_dataset("squad", split=f"validation[{Config.TEST_SIZE + Config.GEN_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE + Config.FORGET_SIZE}]")
    }

datasets = load_squad_splits()
for k, v in datasets.items():
    print(f"{k.capitalize():<10}: {len(v)} samples")

Loading SQuAD Dataset splits...
Train     : 2000 samples
Test      : 200 samples
Gen       : 100 samples
Forget    : 100 samples


In [50]:
print("Loading model and tokenizer via Unsloth...")
max_seq_length = Config.MAX_SEQ_LEN
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = Config.MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype
)

Loading model and tokenizer via Unsloth...
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [51]:
def tokenize_fn(example):
    prompt = f"Context: {example['context'][:Config.CONTEXT_LEN]}\nQuestion: {example['question']}\nAnswer: "
    full_text = prompt + example['answers']['text'][0]

    tokens = tokenizer(full_text, truncation=True, max_length=Config.MAX_SEQ_LEN, padding="max_length")
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=Config.MAX_SEQ_LEN)['input_ids'])

    labels = ([-100] * prompt_len + tokens["input_ids"][prompt_len:])[:Config.MAX_SEQ_LEN]
    labels += [-100] * (Config.MAX_SEQ_LEN - len(labels))
    tokens["labels"] = labels
    return tokens

def normalize_text(text):
    text = text.strip().lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def get_metrics(pred, gold):
    pred, gold = normalize_text(pred), normalize_text(gold)
    em = 1 if pred == gold else 0
    p_tokens, gold_tokens = pred.split(), gold.split()
    if not p_tokens or not gold_tokens: return em, 0.0
    common = set(p_tokens) & set(gold_tokens)
    if not common: return em, 0.0
    prec, rec = len(common)/len(p_tokens), len(common)/len(gold_tokens)
    f1 = 2 * prec * rec / (prec + rec)
    return em, f1

train_tokenized = datasets['train'].map(tokenize_fn, remove_columns=datasets['train'].column_names)
test_tokenized = datasets['test'].map(tokenize_fn, remove_columns=datasets['test'].column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [52]:
def run_evaluation(model, dataset, tokenizer, name="Eval", batch_size=16):
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference
    results = {'em': 0, 'f1': 0, 'acc': 0, 'time': 0}
    print(f"\nEvaluating {name}...")
    
    tokenizer.padding_side = "left" # Left pad for batched generation
    
    prompts = []
    golds = []
    for item in dataset:
        prompt = f"Context: {item['context'][:Config.CONTEXT_LEN]}\nQuestion: {item['question']}\nAnswer:"
        prompts.append(prompt)
        golds.append(item['answers']['text'][0].lower())
        
    t0 = time.time()
    
    all_preds = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=Config.MAX_SEQ_LEN).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                max_length=None,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id
            )
            
        decoded = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for gen in decoded:
            pred = gen.split("\n")[0].strip().lower()
            all_preds.append(pred)
            
    results['time'] = time.time() - t0
    
    for pred, gold in zip(all_preds, golds):
        em, f1 = get_metrics(pred, gold)
        results['em'] += em
        results['f1'] += f1
        if gold in pred or pred in gold: results['acc'] += 1

    count = len(dataset)
    tokenizer.padding_side = "right" # Restore for training
    return {k: (v/count)*100 if k != 'time' else v/count for k, v in results.items()}

In [54]:
import warnings
warnings.filterwarnings("ignore")

In [55]:
print("=== STEP 3: BASE MODEL EVALUATION ===")

base_results = {
    'main': run_evaluation(model, datasets['test'], tokenizer, "Base-Main"),
    'gen': run_evaluation(model, datasets['gen'], tokenizer, "Base-Generalization"),
    'forget': run_evaluation(model, datasets['forget'], tokenizer, "Base-Forgetting")
}

=== STEP 3: BASE MODEL EVALUATION ===

Evaluating Base-Main...

Evaluating Base-Generalization...

Evaluating Base-Forgetting...


In [56]:
base_results

{'main': {'em': 60.0,
  'f1': 70.70120651192632,
  'acc': 74.5,
  'time': 0.5361338460445404},
 'gen': {'em': 56.99999999999999,
  'f1': 64.83308790452753,
  'acc': 71.0,
  'time': 0.7013655543327332},
 'forget': {'em': 63.0,
  'f1': 73.8763316393751,
  'acc': 79.0,
  'time': 0.5999237632751465}}

In [57]:
print("=== STEP 4: LoRA INITIALIZATION ===")

model = FastLanguageModel.get_peft_model(
    model,
    r = Config.LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = Config.LORA_ALPHA,
    lora_dropout = Config.LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = Config.SEED,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

=== STEP 4: LoRA INITIALIZATION ===
trainable params: 20,971,520 || all params: 7,262,703,616 || trainable%: 0.2888


In [58]:
def formatting_prompts_func(example):
    # Handle both single example and batch
    if isinstance(example['question'], list):
        # batched
        output_texts = []
        for i in range(len(example['question'])):
            text = f"Context: {example['context'][i][:Config.CONTEXT_LEN]}\nQuestion: {example['question'][i]}\nAnswer: {example['answers'][i]['text'][0]}"
            output_texts.append(text)
        return output_texts
    else:
        # single example
        text = f"Context: {example['context'][:Config.CONTEXT_LEN]}\nQuestion: {example['question']}\nAnswer: {example['answers']['text'][0]}"
        return [text]

trainer = SFTTrainer(
    model=model,
    train_dataset=datasets['train'],
    eval_dataset=datasets['test'],
    formatting_func=formatting_prompts_func,
    processing_class = tokenizer,
    args=SFTConfig(
        output_dir=Config.OUTPUT_DIR,
        num_train_epochs=Config.EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=4,
        learning_rate=Config.LR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
        seed=Config.SEED,
        max_seq_length=Config.MAX_SEQ_LEN,
        dataset_text_field=None
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

start_time = time.time()
trainer.train()
train_duration = (time.time() - start_time) / 60
model.save_pretrained(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)

Unsloth: Tokenizing ["None"] (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["None"] (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,971,520 of 7,262,703,616 (0.29% trained)


Epoch,Training Loss,Validation Loss
1,0.249368,1.737589


('./lora_model/tokenizer_config.json', './lora_model/tokenizer.json')

In [59]:
print("=== STEP 7: LoRA EVALUATION ===")
lora_results = {
    'main': run_evaluation(model, datasets['test'], tokenizer, "LoRA-Main"),
    'gen': run_evaluation(model, datasets['gen'], tokenizer, "LoRA-Generalization"),
    'forget': run_evaluation(model, datasets['forget'], tokenizer, "LoRA-Forgetting")
}

=== STEP 7: LoRA EVALUATION ===

Evaluating LoRA-Main...

Evaluating LoRA-Generalization...

Evaluating LoRA-Forgetting...


In [62]:
lora_results

{'main': {'em': 53.0,
  'f1': 70.64063630085838,
  'acc': 89.0,
  'time': 0.8264766728878021},
 'gen': {'em': 45.0,
  'f1': 61.50420257417157,
  'acc': 83.0,
  'time': 0.9461582207679748},
 'forget': {'em': 44.0,
  'f1': 65.418281128094,
  'acc': 84.0,
  'time': 0.8535867404937744}}

In [61]:
print("\n" + "="*60)
print("FINAL REFACTORED SUMMARY")
print("="*60)
print(f"Base Acc: {base_results['main']['acc']:.1f}% | LoRA Acc: {lora_results['main']['acc']:.1f}%")
print(f"Base F1 : {base_results['main']['f1']:.1f}% | LoRA F1 : {lora_results['main']['f1']:.1f}%")
print(f"Generalization Delta: {lora_results['gen']['acc'] - base_results['gen']['acc']:+.1f}%")
print(f"Forgetting Delta: {lora_results['forget']['acc'] - base_results['forget']['acc']:+.1f}%")
print(f"Training Time: {train_duration:.1f} min")

with open(Config.RESULTS_FILE, "w") as f:
    json.dump({'base': base_results, 'lora': lora_results}, f, indent=2)


FINAL REFACTORED SUMMARY
Base Acc: 74.5% | LoRA Acc: 89.0%
Base F1 : 70.7% | LoRA F1 : 70.6%
Generalization Delta: +12.0%
Forgetting Delta: +5.0%
Training Time: 28.6 min


In [63]:
import pandas as pd

lora_results = pd.read_json("/kaggle/working/results.json")
lora_results

,base,lora
main,"{'em': 60.0, 'f1': 70.70120651192632, 'acc': 7...","{'em': 53.0, 'f1': 70.64063630085838, 'acc': 8..."
gen,"{'em': 56.99999999999999, 'f1': 64.83308790452...","{'em': 45.0, 'f1': 61.50420257417157, 'acc': 8..."
forget,"{'em': 63.0, 'f1': 73.8763316393751, 'acc': 79...","{'em': 44.0, 'f1': 65.418281128094, 'acc': 84...."
